# How LLMs Work

This notebook walks through the mechanics of a large language model by loading and inspecting
**Qwen3.5-2B** — a 2 billion parameter causal language model from Alibaba.

We'll cover:
1. Loading a pre-trained model
2. Model architecture — the input and output layers (key for optimisations)
3. Tokenization — how text becomes numbers
4. Preparing a chat prompt
5. The forward pass — what the model actually computes
6. Decoding — how numbers become text again
7. Full generation — the autoregressive loop

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## 1. Loading the model

`AutoModelForCausalLM` loads a **causal language model** — one that predicts the next token
given all previous tokens (left-to-right). The model has two components:

- **Tokenizer**: converts text ↔ integer IDs
- **Model weights**: ~2B parameters that encode learned patterns from training data

We load in `float16` to halve memory usage (4GB instead of 8GB).

In [11]:
MODEL_NAME = "Qwen/Qwen3.5-2B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

## 2. Model architecture — input and output layers

Now that we have the model loaded, let's look at what's inside. Understanding these dimensions
is critical for optimisation — every matrix multiply's cost is determined by these shapes.

### The full data flow

```
Input: token IDs [batch, seq_len]        (integers 0–248,319)
                    │
                    ▼
┌─────────────────────────────────────────────────────────┐
│  embed_tokens: Embedding(248320, 2048)                  │  ← INPUT LAYER
│  Maps each token ID to a 2048-dim vector                │
│  Weight matrix: 248,320 × 2,048 = ~509M params (fp16: 1GB)
└─────────────────────────────────────────────────────────┘
                    │
                    ▼  [batch, seq_len, 2048]
┌─────────────────────────────────────────────────────────┐
│  24 × Decoder Layers                                    │
│  (3 linear attention + 1 full attention) × 6 repeats    │
│  Each layer: 2048 → 2048 (hidden size preserved)        │
└─────────────────────────────────────────────────────────┘
                    │
                    ▼  [batch, seq_len, 2048]
┌─────────────────────────────────────────────────────────┐
│  norm: RMSNorm(2048)                                    │
└─────────────────────────────────────────────────────────┘
                    │
                    ▼  [batch, seq_len, 2048]
┌─────────────────────────────────────────────────────────┐
│  lm_head: Linear(2048, 248320)                          │  ← OUTPUT LAYER
│  Projects hidden state back to vocabulary logits         │
│  Weight matrix: 2,048 × 248,320 = ~509M params          │
│  (TIED with embed_tokens — same weight matrix transposed)│
└─────────────────────────────────────────────────────────┘
                    │
                    ▼
Output: logits [batch, seq_len, 248320]  (raw scores per token)
```

### Why this matters for optimisation

**Input layer (`embed_tokens`)**: A simple table lookup — O(1) per token. No matrix multiply.
The 509M parameters live in memory but the forward pass is cheap (just indexing rows).

**Output layer (`lm_head`)**: The most expensive single operation per forward pass.
It multiplies `[batch × seq_len, 2048] × [2048, 248320]` — that's a matmul with 248K output
features. For a sequence of 1000 tokens, that's ~509 billion FLOPs in a single layer.
This is a key target for optimisation (vocabulary pruning, speculative decoding, etc).

**Weight tying**: The embedding and lm_head share the same weight matrix (transposed).
This saves ~1GB of memory but means you can't quantize them independently.

| Component | Dimension |
|-----------|----------|
| Vocabulary | 248,320 tokens |
| Embedding / Hidden size | 2,048 |
| Layers | 24 (repeating: 3 linear attention + 1 full attention) |
| Attention heads | 8 query, 2 key/value (Grouped Query Attention) |
| Head dimension | 256 |
| MLP intermediate | 6,144 (3× expansion) |
| Max context | 262,144 tokens (256K) |
| Position encoding | Multimodal RoPE (3 axes: temporal, height, width) |

In [12]:
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Embedding params: {model.model.embed_tokens.weight.numel():,} ({model.model.embed_tokens.weight.numel()/sum(p.numel() for p in model.parameters())*100:.1f}%)")
print(f"Memory (fp16): {sum(p.numel() for p in model.parameters()) * 2 / 1e9:.2f} GB")

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2048)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (in_proj_qkv): Linear(in_features=2048, out_features=6144, bias=False)
          (in_proj_z): Linear(in_features=2048, out_features=2048, bias=False)
          (in_proj_b): Linear(in_features=2048, out_features=16, bias=False)
          (in_proj_a): Linear(in_features=2048, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(

## 3. Tokenization — how text becomes numbers

Neural networks can't process text directly. The tokenizer splits text into **tokens** (subwords,
words, or character fragments) and maps each to an integer ID.

Qwen uses **Byte-Pair Encoding (BPE)** with a vocabulary of 248,320 tokens. This includes:
- Common English words: `Everyone`, `side`
- Multilingual tokens: Chinese, Korean, Cyrillic, Portuguese
- Subword fragments: `-Version`, `-side`
- Special tokens: `<think>`, role markers

The `Ġ` prefix represents a leading space (the space gets attached to the following word).

In [13]:
# Peek at 10 tokens from the vocabulary
{v: k for k, v in list(tokenizer.get_vocab().items())[:10]}

{210351: 'Ġbasado',
 76637: 'iens',
 3378: 'sv',
 144406: 'æĬ¢çľ¼',
 79070: '.WebServlet',
 13046: 'Ġtalks',
 193687: 'ĠÑģÐ¿Ð¾ÑģÐ¾Ð±ÐµÐ½',
 163755: 'ãģ¨ãĤĬ',
 73133: 'Ġexceedingly',
 184625: 'Ġà¸ªà¸£à¹īà¸²à¸ĩ'}

In [14]:
# Tokenize a simple sentence and see the IDs
example = "Hello, world!"
token_ids = tokenizer.encode(example)
tokens = [tokenizer.decode(tid) for tid in token_ids]

print(f"Text:      {example!r}")
print(f"Token IDs: {token_ids}")
print(f"Tokens:    {tokens}")
print(f"Count:     {len(token_ids)} tokens")

Text:      'Hello, world!'
Token IDs: [9419, 11, 1814, 0]
Tokens:    ['Hello', ',', ' world', '!']
Count:     4 tokens


## 4. Preparing a chat prompt

Chat-tuned models expect a specific format with role markers (system, user, assistant).
`apply_chat_template` wraps our prompt in the model's expected structure and adds a
generation prompt so the model knows it should start responding.

The tokenizer then converts this formatted string into:
- `input_ids`: tensor of token IDs (shape `[batch, seq_len]`)
- `attention_mask`: tensor of 1s indicating which positions are real tokens (vs padding)

In [15]:
prompt = "Explain what a large language model is in one sentence."

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Formatted prompt:")
print(repr(text))

Formatted prompt:
'<|im_start|>user\nExplain what a large language model is in one sentence.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [7]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)
print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Token IDs:   {inputs['input_ids'][0].tolist()}")

Input shape: torch.Size([1, 24])
Token IDs:   [248045, 846, 198, 814, 20139, 1092, 264, 3349, 3992, 1558, 369, 303, 799, 11316, 13, 248046, 198, 248045, 74455, 198, 248068, 271, 248069, 271]


## 5. The forward pass — what the model actually computes

A single forward pass takes the input token IDs and produces **logits** — a raw score for
every token in the vocabulary, at every position in the sequence.

Referring back to our architecture diagram, the input tokens flow through all 24 layers
and emerge as a `[batch, seq_len, 248320]` tensor of logits — one score per vocabulary
token, per position.

In [8]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f"Logits shape: {logits.shape}")
print(f"  - Batch size: {logits.shape[0]}")
print(f"  - Sequence length: {logits.shape[1]} tokens")
print(f"  - Vocabulary size: {logits.shape[2]} possible next tokens")

Logits shape: torch.Size([1, 24, 248320])
  - Batch size: 1
  - Sequence length: 24 tokens
  - Vocabulary size: 248320 possible next tokens


## 6. Decoding — turning logits into text

To predict the **next** token, we only care about the logits at the **last position**
(the model's prediction for what comes after the entire input).

### Greedy decoding
The simplest strategy: pick the token with the highest logit score (argmax).

In [36]:
last_logits = logits[0, -1, :]  # shape: [248320]
next_token_id = last_logits.argmax()

print(f"Highest-scoring token ID: {next_token_id.item()}")
print(f"Decoded: {tokenizer.decode(next_token_id)!r}")

Highest-scoring token ID: 32
Decoded: 'A'


### Top-K: what else did the model consider?

Looking at only the argmax hides the model's uncertainty. The top-K tokens show the
full picture — the model assigns probability mass across many plausible continuations.

Sampling strategies (temperature, top-p, top-k) choose among these candidates to add
diversity. Higher temperature → more random; lower → more deterministic.

In [37]:
top_k = torch.topk(last_logits, k=10)

# Convert logits to probabilities with softmax
probs = torch.softmax(last_logits, dim=-1)
top_probs = probs[top_k.indices]

print("Top 10 next-token candidates:")
print(f"{'Rank':<5} {'Token':<12} {'Probability':<12}")
print("-" * 30)
for i, (idx, prob) in enumerate(zip(top_k.indices, top_probs)):
    token = tokenizer.decode(idx)
    print(f"{i+1:<5} {token:<12} {prob:.4f}")

Top 10 next-token candidates:
Rank  Token        Probability 
------------------------------
1     A            0.9922
2     Large        0.0067
3     An           0.0009
4     It           0.0003
5     In           0.0001
6     a            0.0001
7     The          0.0001
8     <think>      0.0001
9     Art          0.0000
10    L            0.0000


## 7. Full generation — the autoregressive loop

Generation is just the forward pass + decoding repeated in a loop:

1. Run forward pass → get logits for next token
2. Pick a token (greedy, sampling, beam search, etc.)
3. Append that token to the input
4. Repeat until a stop token or max length

`model.generate()` handles this loop with KV-caching for efficiency (so we don't
recompute attention over the full history every step).

In [38]:
output_ids = model.generate(**inputs, max_new_tokens=128)

# Decode only the generated tokens (skip the input)
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nResponse: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Prompt: Explain what a large language model is in one sentence.

Response: A large language model is a sophisticated type of artificial intelligence trained on vast amounts of text data to understand, generate, and reason about human language.

